In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df = pd.read_csv('cars.csv')
df.sample(5)

,brand,km_driven,fuel,owner,selling_price
6623,Maruti,80000,Diesel,Fourth & Above Owner,375000
2306,Chevrolet,90000,Diesel,Second Owner,185000
1598,Hyundai,30000,Petrol,First Owner,520000
5301,Maruti,50000,Petrol,First Owner,482000
2520,Maruti,60000,Diesel,First Owner,610000


In [3]:
df['owner'].value_counts()

owner
First Owner             5289
Second Owner            2105
Third Owner              555
Fourth & Above Owner     174
Test Drive Car             5
Name: count, dtype: int64

In [4]:
# OneHotEncoding using Pandas:
pd.get_dummies(df,columns=['fuel','owner'])
# but this will not do permanent changes to the dataframe.

,brand,km_driven,selling_price,fuel_CNG,fuel_Diesel,fuel_LPG,fuel_Petrol,owner_First Owner,owner_Fourth & Above Owner,owner_Second Owner,owner_Test Drive Car,owner_Third Owner
0,Maruti,145500,450000,False,True,False,False,True,False,False,False,False
1,Skoda,120000,370000,False,True,False,False,False,False,True,False,False
2,Honda,140000,158000,False,False,False,True,False,False,False,False,True
3,Hyundai,127000,225000,False,True,False,False,True,False,False,False,False
4,Maruti,120000,130000,False,False,False,True,True,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...
8123,Hyundai,110000,320000,False,False,False,True,True,False,False,False,False
8124,Hyundai,119000,135000,False,True,False,False,False,True,False,False,False
8125,Maruti,120000,382000,False,True,False,False,True,False,False,False,False
8126,Tata,25000,290000,False,True,False,False,True,False,False,False,False


In [5]:
# K-1 OneHotEncoding : (if we have n categories, we can represent them using n-1 binary variables to avoid multicollinearity)
pd.get_dummies(df,columns=['fuel','owner'],drop_first=True)


,brand,km_driven,selling_price,fuel_Diesel,fuel_LPG,fuel_Petrol,owner_Fourth & Above Owner,owner_Second Owner,owner_Test Drive Car,owner_Third Owner
0,Maruti,145500,450000,True,False,False,False,False,False,False
1,Skoda,120000,370000,True,False,False,False,True,False,False
2,Honda,140000,158000,False,False,True,False,False,False,True
3,Hyundai,127000,225000,True,False,False,False,False,False,False
4,Maruti,120000,130000,False,False,True,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...
8123,Hyundai,110000,320000,False,False,True,False,False,False,False
8124,Hyundai,119000,135000,True,False,False,True,False,False,False
8125,Maruti,120000,382000,True,False,False,False,False,False,False
8126,Tata,25000,290000,True,False,False,False,False,False,False


In [6]:
# OneHotEncoding using sklearn:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(df.iloc[:,0:4],df.iloc[:,-1],test_size=0.2,random_state=2)
X_train.shape, y_train.shape

((6502, 4), (6502,))

In [8]:
X_train.sample(5)

,brand,km_driven,fuel,owner
6895,Mahindra,40000,Diesel,First Owner
2603,Hyundai,4000,Petrol,First Owner
7617,Renault,35000,Petrol,First Owner
2429,Tata,25000,Petrol,First Owner
4434,Toyota,167000,Petrol,First Owner


In [9]:
y_train

5571    1150000
2038    1689999
2957     580000
7618     150000
6684     320000
         ...   
3606     620000
5704     335000
6637     450000
2575     651000
7336    1160000
Name: selling_price, Length: 6502, dtype: int64

In [11]:
from sklearn.preprocessing import OneHotEncoder
ohe = OneHotEncoder(drop='first',sparse_output=False,dtype=np.int32)
X_train_ohe = ohe.fit_transform(X_train[['fuel','owner']])
X_train_ohe = ohe.transform(X_train[['fuel','owner']])
X_test_ohe = ohe.transform(X_test[['fuel','owner']])
X_train_ohe.shape, X_test_ohe.shape

((6502, 7), (1626, 7))

In [16]:
X_train_ohe = np.hstack((X_train[['brand','km_driven']].values,X_train_ohe))

In [18]:
X_train_ohe

array([['Hyundai', 35000, 1, ..., 0, 0, 0],
       ['Jeep', 60000, 1, ..., 0, 0, 0],
       ['Hyundai', 25000, 0, ..., 0, 0, 0],
       ...,
       ['Tata', 15000, 0, ..., 0, 0, 0],
       ['Maruti', 32500, 1, ..., 1, 0, 0],
       ['Isuzu', 121000, 1, ..., 0, 0, 0]], shape=(6502, 9), dtype=object)

In [20]:
X_train_ohe_df = pd.DataFrame(X_train_ohe,columns=['brand','km_driven']+list(ohe.get_feature_names_out(['fuel','owner'])))
X_train_ohe_df.head()

,brand,km_driven,fuel_Diesel,fuel_LPG,fuel_Petrol,owner_Fourth & Above Owner,owner_Second Owner,owner_Test Drive Car,owner_Third Owner
0,Hyundai,35000,1,0,0,0,0,0,0
1,Jeep,60000,1,0,0,0,0,0,0
2,Hyundai,25000,0,0,1,0,0,0,0
3,Mahindra,130000,1,0,0,0,1,0,0
4,Hyundai,155000,1,0,0,0,0,0,0


In [25]:
#OneHalfEncoding with Top Categories:
counts = df['brand'].value_counts()
counts

brand
Maruti           2448
Hyundai          1415
Mahindra          772
Tata              734
Toyota            488
Honda             467
Ford              397
Chevrolet         230
Renault           228
Volkswagen        186
BMW               120
Skoda             105
Nissan             81
Jaguar             71
Volvo              67
Datsun             65
Mercedes-Benz      54
Fiat               47
Audi               40
Lexus              34
Jeep               31
Mitsubishi         14
Land                6
Force               6
Isuzu               5
Ambassador          4
Kia                 4
MG                  3
Daewoo              3
Ashok               1
Opel                1
Peugeot             1
Name: count, dtype: int64

In [ ]:
df['brand'].nunique()
threshold =100


In [28]:
repl = counts[counts<= threshold].index
repl

Index(['Nissan', 'Jaguar', 'Volvo', 'Datsun', 'Mercedes-Benz', 'Fiat', 'Audi',
       'Lexus', 'Jeep', 'Mitsubishi', 'Land', 'Force', 'Isuzu', 'Ambassador',
       'Kia', 'MG', 'Daewoo', 'Ashok', 'Opel', 'Peugeot'],
      dtype='object', name='brand')

In [29]:
df['brand'].replace(repl,'other',inplace=True)

In [32]:
pd.get_dummies(df['brand'],drop_first=True,dtype=int)

,Chevrolet,Ford,Honda,Hyundai,Mahindra,Maruti,Renault,Skoda,Tata,Toyota,Volkswagen,other
0,0,0,0,0,0,1,0,0,0,0,0,0
1,0,0,0,0,0,0,0,1,0,0,0,0
2,0,0,1,0,0,0,0,0,0,0,0,0
3,0,0,0,1,0,0,0,0,0,0,0,0
4,0,0,0,0,0,1,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...
8123,0,0,0,1,0,0,0,0,0,0,0,0
8124,0,0,0,1,0,0,0,0,0,0,0,0
8125,0,0,0,0,0,1,0,0,0,0,0,0
8126,0,0,0,0,0,0,0,0,1,0,0,0
